In [ ]:
import polars as pl
# Lê apenas a primeira linha para ver o "cabeçalho"
schema = pl.read_parquet("statsbomb_completo.parquet", n_rows=1).columns
print("📋 Colunas reais do arquivo:", schema)

In [18]:
import polars as pl
import numpy as np
import os

# Garantir que a pasta de destino existe
os.makedirs("Data/Gold", exist_ok=True)

print("⚡ Carregando e processando dados (1M linhas)...")

# 1. CARGA SELETIVA
df = pl.read_parquet("statsbomb_completo.parquet", n_rows=1000000)
df = df.rename({col: col.replace(".", "_") for col in df.columns}).sort(["match_id", "index"])

# 2. FEATURES GEOMÉTRICAS
df = df.with_columns([
    pl.col("location").list.get(0).fill_null(120.0).alias("x"),
    pl.col("location").list.get(1).fill_null(40.0).alias("y"),
    pl.col("under_pressure").fill_null(False).cast(pl.Int8).alias("pressao_binaria")
])

x_arr, y_arr = df["x"].to_numpy(), df["y"].to_numpy()
dist = np.sqrt((120 - x_arr)**2 + (40 - y_arr)**2)

# Lei dos Cossenos para o ângulo de visão
a = np.sqrt((120 - x_arr)**2 + (36 - y_arr)**2)
b = np.sqrt((120 - x_arr)**2 + (44 - y_arr)**2)
cos_theta = np.clip((a**2 + b**2 - 8**2) / (2 * a * b), -1.0, 1.0)
angulo = np.degrees(np.arccos(cos_theta))

# 3. FEATURES TEMPORAIS E CONTEXTO
df_contexto = df.with_columns([
    pl.Series("distancia", dist),
    pl.Series("angulo_visao", angulo),
    pl.col("pressao_binaria").rolling_mean(window_size=10, min_samples=1).over("match_id").alias("pressao_10_lances"),
    (pl.col("x") - pl.col("x").shift(5).over("match_id")).alias("aceleracao_ataque"),
    (pl.col("x") - pl.col("x").shift(1).over("match_id")).alias("progresso_ultimo_lance")
])

# 4. TARGET: Iminência (Janela de 10 eventos)
df_total = df_contexto.with_columns([
    (pl.col("type_name") == "Shot").cast(pl.Int8).alias("eh_chute")
]).with_columns([
    pl.col("eh_chute")
      .shift(-10)
      .rolling_max(window_size=10, min_samples=1)
      .over("match_id")
      .fill_null(0)
      .alias("iminencia_gol")
])

# ====================== 5. O SPLIT ESTRATÉGICO ======================

# --- DATASET 1: ATAQUE (O Filé Mignon para o XGBoost) ---
features_treino = [
    "match_id", "player_name", "x", "y", "distancia", 
    "angulo_visao", "pressao_10_lances", "aceleracao_ataque", 
    "progresso_ultimo_lance", "iminencia_gol"
]

df_ataque = df_total.filter(
    pl.col("type_name").is_in(["Pass", "Carry", "Dribble", "Ball Recovery"])
).select(features_treino).fill_null(0)

# --- DATASET 2: DEFESA/GOLEIRO (O "Dicionário" de resultados) ---
# Pegamos colunas que explicam por que um gol NÃO aconteceu (bloqueios e defesas)
cols_defesa = ["match_id", "index", "player_name", "type_name"] + \
              [c for c in df_total.columns if "goalkeeper" in c or "block" in c or "foul" in c]

df_defesa = df_total.select(cols_defesa).filter(
    pl.col("type_name").is_in(["Goalkeeper", "Block", "Foul Committed", "Foul Won"])
)

# 6. SALVANDO OS RESULTADOS
df_ataque.write_parquet("Data/Gold/ataque_oraculo.parquet")
df_defesa.write_parquet("Data/Gold/defesa_lookup.parquet")

print("-" * 30)
print(f"✅ DATASET ATAQUE: {df_ataque.height:,} linhas | {df_ataque.width} colunas")
print(f"🛡️ DATASET DEFESA: {df_defesa.height:,} linhas | {df_defesa.width} colunas")
print(f"📊 Taxa de Iminência: {df_ataque['iminencia_gol'].mean():.2%}")
print("-" * 30)

⚡ Carregando e processando dados (1M linhas)...
------------------------------
✅ DATASET ATAQUE: 543,005 linhas | 10 colunas
🛡️ DATASET DEFESA: 23,953 linhas | 37 colunas
📊 Taxa de Iminência: 6.85%
------------------------------


In [20]:
df_ataque.head(15)

match_id,player_name,x,y,distancia,angulo_visao,pressao_10_lances,aceleracao_ataque,progresso_ultimo_lance,iminencia_gol
str,str,f64,f64,f64,f64,f64,f64,f64,i8
"""15946""","""Jonathan Rodríguez Menéndez""",61.0,40.1,59.000085,7.757027,0.0,0.0,-59.0,0
"""15946""","""Guillermo Alfonso Maripán Loay…",33.8,28.0,87.031259,5.212992,0.0,-86.2,0.0,0
"""15946""","""Guillermo Alfonso Maripán Loay…",36.8,27.3,84.16371,5.380086,0.0,-83.2,3.0,0
"""15946""","""Sergio Busquets i Burgos""",33.6,5.9,92.88579,4.588851,0.2,-0.2,-52.9,0
"""15946""","""Ivan Rakitić""",35.1,18.3,87.629333,5.065166,0.2,-1.7,0.0,0
…,…,…,…,…,…,…,…,…,…
"""15946""","""Víctor Laguardia Cisneros""",35.8,69.7,89.284545,4.839607,0.0,1.5,5.8,0
"""15946""","""Marc-André ter Stegen""",13.7,24.3,107.453153,4.218171,0.0,-11.6,-22.1,0
"""15946""","""Marc-André ter Stegen""",13.7,24.3,107.453153,4.218171,0.0,-12.0,0.0,0


In [16]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("🚀 Iniciando treinamento do Oráculo...")

df_ml = df_final.to_pandas()

# Lista de ingredientes confirmada
features = [
    'distancia', 'angulo_visao', 'pressao_10_lances', 
    'aceleracao_ataque', 'x', 'y'
]

X = df_ml[features]
y = df_ml['iminencia_gol']

# Cálculo de peso dinâmico (Ouro para AUC alto)
peso_calibrado = len(y[y==0]) / len(y[y==1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_oraculo = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=peso_calibrado, # Agora o modelo vai valorizar o perigo
    tree_method='hist',
    eval_metric='auc'
)

model_oraculo.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# Validação final
preds = model_oraculo.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, preds)

print(f"🔥 SUCESSO! Novo AUC: {auc:.4f}")
model_oraculo.save_model("oraculo_iminencia.json")

🚀 Iniciando treinamento do Oráculo...
🔥 SUCESSO! Novo AUC: 0.7791


testar

In [8]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import shap
import matplotlib.pyplot as plt

print("🚀 Iniciando PIPELINE PROFISSIONAL do Oráculo...")

df_ml = df_final.to_pandas()

# ====================== FEATURES (agora com mais poder) ======================
features = [
    'distancia', 'angulo_visao', 'pressao_10_lances', 
    'aceleracao_ataque', 'x', 'y', 'progresso_ultimo_lance'
]

# (Opcional: você pode adicionar 'acao_anterior' com OneHot se quiser depois)

X = df_ml[features].values
y = df_ml['iminencia_gol'].values
groups = df_ml['match_id'].values

# Peso para desbalanceamento
peso_calibrado = len(y[y==0]) / len(y[y==1])

# ====================== VALIDAÇÃO CORRETA: GroupKFold ======================
gkf = GroupKFold(n_splits=5)
auc_scores = []
models = []
shap_values_list = []

print("🔄 Rodando 5-fold GroupKFold (por partida)...")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=800,
        max_depth=7,
        learning_rate=0.04,
        scale_pos_weight=peso_calibrado,
        tree_method='hist',
        eval_metric='auc',
        early_stopping_rounds=60,
        random_state=42 + fold
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, pred)
    auc_scores.append(auc)
    models.append(model)
    
    # SHAP (apenas no último fold para não demorar muito)
    if fold == 5:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        shap_values_list = shap_values  # salva para plot
    
    print(f"   Fold {fold} → AUC: {auc:.4f}")

print("\n" + "="*50)
print(f"✅ AUC MÉDIO (GroupKFold): {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*50)

# ====================== BASELINE (só distância) ======================
from sklearn.linear_model import LogisticRegression

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X[:, 0:1], y)  # só 'distancia'
pred_base = baseline.predict_proba(X[:, 0:1])[:, 1]
auc_base = roc_auc_score(y, pred_base)
print(f"📉 AUC Baseline (apenas distância): {auc_base:.4f}")

# ====================== FEATURE IMPORTANCE + SHAP ======================
best_model = models[-1]  # último modelo (melhor prática)

plt.figure(figsize=(10, 6))
xgb.plot_importance(best_model, importance_type='gain', max_num_features=10, title="Feature Importance (Gain)")
plt.show()

# SHAP summary (último fold)
shap.summary_plot(shap_values_list, X_test, feature_names=features, show=False)
plt.title("SHAP Summary Plot - Oráculo de Iminência")
plt.show()

# ====================== SALVAMENTO ======================
best_model.save_model("oraculo_iminencia_v2.json")
print("💾 Modelo salvo como 'oraculo_iminencia_v2.json'")

# Relatório final
print("\n📊 RELATÓRIO FINAL")
print(f"• AUC médio real: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"• Melhoria vs baseline: +{np.mean(auc_scores) - auc_base:.4f}")
print(f"• Total de partidas únicas: {len(np.unique(groups)):,}")
print("✅ Pipeline pronto para produção!")

ModuleNotFoundError: No module named 'shap'